# 04 - Feature Engineering

## Objetivo

Construir o dataset final de modelagem em nível `user_id-product_id` para o sistema de recomendação de próximo carrinho.

Este notebook parte dos candidatos e targets definidos no notebook anterior e adiciona features históricas calculadas exclusivamente a partir do conjunto `prior`.

Este notebook cobre:

- Carregamento dos candidatos supervisionados gerados no notebook `03-candidate-strategy`.
- Cálculo de features históricas em nível de usuário.
- Cálculo de features históricas em nível de produto.
- Cálculo de features históricas em nível usuário-produto.
- Cálculo de features históricas em nível de categoria.
- Consolidação incremental do dataset final.
- Tratamento explícito de valores ausentes.
- Validação anti-leakage.
- Persistência do dataset final de modelagem.

## Inputs

- `data/processed/orders_product_unified.parquet`
- `data/features/candidates_v2_with_target.parquet`

## Outputs esperados

- `data/features/modeling_dataset_v2/` — dataset final particionado em Parquet

## Limites deste notebook

Este notebook não gera candidatos.

Este notebook não reconstrói o target supervisionado.

Este notebook não treina modelos.

Este notebook não implementa baselines.

Este notebook não define a arquitetura MLP.

Todas as features são calculadas exclusivamente a partir do conjunto `prior`.

O conjunto `train` aparece apenas indiretamente no arquivo de candidatos com target produzido pelo notebook anterior. Nenhuma coluna derivada de `train` é usada como feature.

O dataset final pode ser salvo como diretório Parquet particionado, em vez de arquivo único, para reduzir uso de memória e facilitar leitura incremental na etapa de modelagem.

---

## 1. Setup inicial

In [1]:
import gc
from math import ceil
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pyarrow.compute as pc
import pyarrow.dataset as ds

pd.set_option("display.max_columns", None)

In [2]:
PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
TEST_SAMPLE_DIR = DATA_DIR / "test_sample"
FEATURES_DIR = DATA_DIR / "features"

UNIFIED_DATASET_PATH = PROCESSED_DIR / "orders_product_unified.parquet"
SAMPLE_DATASET_PATH = TEST_SAMPLE_DIR / "orders_product_sample.parquet"


CANDIDATES_PATH = FEATURES_DIR / "candidates_v2_with_target.parquet"

STAGED_MODELING_DATASET_DIR = FEATURES_DIR / "modeling_dataset_v2_raw"
MODELING_DATASET_DIR = FEATURES_DIR / "modeling_dataset_v2"


for input_path in [UNIFIED_DATASET_PATH, CANDIDATES_PATH]:
    assert input_path.exists(), f"Arquivo de entrada não encontrado: {input_path}"


print("Arquivos de entrada encontrados com sucesso.")
print(f"Dataset final será salvo em: {MODELING_DATASET_DIR}")

Arquivos de entrada encontrados com sucesso.
Dataset final será salvo em: /Users/helio/Library/CloudStorage/OneDrive-Pessoal/Pos_FIAP/Tech_Challanges/Fase2-Big-Data-Architecture/mlp-market-recommender-system/data/features/modeling_dataset_v2


### 1.1 Carregamento dos dados

In [3]:
USE_SAMPLE = False

dataset_path = SAMPLE_DATASET_PATH if USE_SAMPLE else UNIFIED_DATASET_PATH

df = pd.read_parquet(dataset_path)
modeling_df = pd.read_parquet(CANDIDATES_PATH)

print(f"Dataset unificado carregado: {dataset_path.name}")
print(f"df: {df.shape[0]:,} linhas x {df.shape[1]:,} colunas")
print(f"modeling_df: {modeling_df.shape[0]:,} linhas x {modeling_df.shape[1]:,} colunas")

if USE_SAMPLE:
    print(
        "\nATENÇÃO: amostra auxiliar carregada. "
        "Não use este resultado para conclusões finais."
    )

Dataset unificado carregado: orders_product_unified.parquet
df: 33,819,106 linhas x 15 colunas
modeling_df: 45,519,000 linhas x 3 colunas


## 2 Preparação do universo de features

In [4]:
df_prior = df[df["eval_set"] == "prior"].copy()

eligible_users = set(modeling_df["user_id"].unique())
prior_users = set(df_prior["user_id"].unique())

assert eligible_users.issubset(prior_users), (
    "Existem candidatos para usuários sem histórico prior."
)
assert modeling_df["target"].isin([0, 1]).all(), (
    "A coluna target contém valores fora de {0, 1}."
)
assert not modeling_df.duplicated(subset=["user_id", "product_id"]).any(), (
    "Existem pares user_id-product_id duplicados em modeling_df."
)
assert df_prior["eval_set"].eq("prior").all(), (
    "df_prior contém registros fora do conjunto prior."
)

print(f"df_prior: {df_prior.shape[0]:,} linhas")
print(f"Usuários elegíveis: {len(eligible_users):,}")
print(f"Pares candidatos: {len(modeling_df):,}")
print(f"Positivos: {modeling_df['target'].sum():,}")

df_prior: 32,434,489 linhas
Usuários elegíveis: 131,209
Pares candidatos: 45,519,000
Positivos: 991,122


---

## 3. Features de usuário

Esta seção calcula features históricas em nível `user_id`, usando exclusivamente o conjunto `prior`.

As features representam profundidade de histórico, tamanho médio de carrinho, comportamento de recompra e frequência média entre pedidos.

A imputação de `user_avg_days_between_orders` é necessária para usuários sem intervalo histórico observável suficiente.

In [5]:
prior_user_data = df_prior[df_prior["user_id"].isin(eligible_users)].copy()

prior_order_basket_size = (
    prior_user_data
    .groupby(["user_id", "order_id"], as_index=False)
    .agg(
        basket_size=("product_id", "count"),
        days_since_prior_order=("days_since_prior_order", "first"),
    )
)

user_order_features = (
    prior_order_basket_size
    .groupby("user_id", as_index=False)
    .agg(
        user_prior_order_count=("order_id", "nunique"),
        user_avg_basket_size=("basket_size", "mean"),
        user_avg_days_between_orders=("days_since_prior_order", "mean"),
    )
)

user_reorder_features = (
    prior_user_data
    .groupby("user_id", as_index=False)
    .agg(
        user_reorder_rate=("reordered", "mean"),
    )
)

user_features = user_order_features.merge(
    user_reorder_features,
    on="user_id",
    how="inner",
)

user_features["user_has_single_prior_order"] = (
    user_features["user_avg_days_between_orders"].isna().astype(int)
)

user_features["user_avg_days_between_orders"] = (
    user_features["user_avg_days_between_orders"]
    .fillna(0)
)

assert user_features["user_id"].is_unique, "user_features contém usuários duplicados."
assert set(user_features["user_id"]) == eligible_users, (
    "user_features não cobre exatamente os usuários elegíveis."
)
assert user_features["user_prior_order_count"].ge(1).all(), (
    "Existem usuários sem pedidos prior."
)
assert user_features["user_avg_basket_size"].gt(0).all(), (
    "Existem usuários com tamanho médio de carrinho inválido."
)
assert user_features["user_reorder_rate"].between(0, 1).all(), (
    "user_reorder_rate contém valores fora do intervalo [0, 1]."
)
assert user_features["user_avg_days_between_orders"].ge(0).all(), (
    "user_avg_days_between_orders contém valores negativos."
)

print(f"user_features: {user_features.shape[0]:,} usuários x {user_features.shape[1]:,} colunas")
print(f"Valor para usuários sem intervalo prior observável: {0}")
print(f"Usuários sem intervalo prior observável: {user_features['user_has_single_prior_order'].sum():,}")

user_features: 131,209 usuários x 6 colunas
Valor para usuários sem intervalo prior observável: 0
Usuários sem intervalo prior observável: 0


In [6]:
user_features.head()

,user_id,user_prior_order_count,user_avg_basket_size,user_avg_days_between_orders,user_reorder_rate,user_has_single_prior_order
0,1,10,5.900000,19.555556,0.694915,0
1,2,14,13.928571,15.230769,0.476923,0
2,5,4,9.250000,13.333333,0.378378,0
3,7,20,10.300000,10.684211,0.669903,0
4,8,3,16.333333,30.000000,0.265306,0


---

## 4. Features de produto

Esta seção calcula features históricas em nível `product_id`, usando exclusivamente o conjunto `prior`.

As features representam popularidade absoluta, taxa histórica de recompra e versões normalizadas ou transformadas da popularidade para reduzir o efeito da escala bruta de compras.

In [7]:
total_prior_purchases = len(df_prior)

product_features = (
    df_prior
    .groupby("product_id", as_index=False)
    .agg(
        product_prior_purchase_count=("order_id", "size"),
        product_prior_reorder_rate=("reordered", "mean"),
    )
)

product_features["product_popularity_pct"] = (
    product_features["product_prior_purchase_count"] / total_prior_purchases
)

product_features["product_popularity_log"] = np.log1p(
    product_features["product_prior_purchase_count"]
)

assert product_features["product_id"].is_unique, (
    "product_features contém produtos duplicados."
)
assert product_features["product_prior_purchase_count"].gt(0).all(), (
    "Existem produtos sem compras no prior."
)
assert product_features["product_prior_reorder_rate"].between(0, 1).all(), (
    "product_prior_reorder_rate contém valores fora do intervalo [0, 1]."
)
assert product_features["product_popularity_pct"].between(0, 1).all(), (
    "product_popularity_pct contém valores fora do intervalo [0, 1]."
)
assert np.isclose(
    product_features["product_popularity_pct"].sum(),
    1.0,
), "A soma de product_popularity_pct deveria ser aproximadamente 1."

print(f"product_features: {product_features.shape[0]:,} produtos x {product_features.shape[1]:,} colunas")
print(f"Total de compras no prior: {total_prior_purchases:,}")

product_features: 49,677 produtos x 5 colunas
Total de compras no prior: 32,434,489


In [8]:
product_features.head()

,product_id,product_prior_purchase_count,product_prior_reorder_rate,product_popularity_pct,product_popularity_log
0,1,1852,0.613391,5.709971e-05,7.524561
1,2,90,0.133333,2.774824e-06,4.510860
2,3,277,0.732852,8.540292e-06,5.627621
3,4,329,0.446809,1.014352e-05,5.799093
4,5,15,0.600000,4.624707e-07,2.772589


---

## 5. Features usuário-produto

Esta seção calcula features históricas em nível `user_id-product_id`, usando exclusivamente o conjunto `prior`.

Essas features capturam a relação direta entre cada usuário e cada produto: frequência de compra, recompra, posição média no carrinho e recência da última compra.

A recência em dias é validada separadamente porque depende da ordenação temporal dos pedidos e pode gerar bugs silenciosos caso o cálculo use linhas de produto em vez de pedidos únicos.

In [9]:
df_prior_ordered = (
    df_prior[df_prior["user_id"].isin(eligible_users)]
    .sort_values(["user_id", "order_number", "add_to_cart_order"])
    .copy()
)

assert df_prior_ordered[["user_id", "order_number"]].notna().all().all(), (
    "Existem valores nulos em user_id ou order_number."
)
assert df_prior_ordered["eval_set"].eq("prior").all(), (
    "df_prior_ordered contém registros fora do prior."
)

print(f"df_prior_ordered: {df_prior_ordered.shape[0]:,} linhas")
print(f"Usuários elegíveis cobertos: {df_prior_ordered['user_id'].nunique():,}")

df_prior_ordered: 20,641,991 linhas
Usuários elegíveis cobertos: 131,209


In [10]:
user_last_prior_order = (
    df_prior_ordered
    .groupby("user_id", as_index=False)
    .agg(user_last_prior_order_number=("order_number", "max"))
)

user_product_features = (
    df_prior_ordered
    .groupby(["user_id", "product_id"], as_index=False)
    .agg(
        user_product_purchase_count=("order_id", "nunique"),
        user_product_reorder_count=("reordered", "sum"),
        user_product_avg_add_to_cart_order=("add_to_cart_order", "mean"),
        user_product_last_order_number=("order_number", "max"),
    )
    .merge(
        user_last_prior_order,
        on="user_id",
        how="left",
    )
)

user_product_features["user_product_was_bought_before"] = 1

user_product_features["user_product_orders_since_last_purchase"] = (
    user_product_features["user_last_prior_order_number"]
    - user_product_features["user_product_last_order_number"]
)

user_product_features = user_product_features.drop(
    columns=["user_last_prior_order_number", "user_product_last_order_number"]
)

assert not user_product_features.duplicated(subset=["user_id", "product_id"]).any(), (
    "user_product_features contém pares user_id-product_id duplicados."
)
assert user_product_features["user_product_purchase_count"].ge(1).all(), (
    "Existem pares históricos com contagem de compra menor que 1."
)
assert user_product_features["user_product_reorder_count"].ge(0).all(), (
    "Existem contagens de recompra negativas."
)
assert user_product_features["user_product_orders_since_last_purchase"].ge(0).all(), (
    "orders_since_last_purchase contém valores negativos."
)

print(f"user_product_features: {user_product_features.shape[0]:,} pares x {user_product_features.shape[1]:,} colunas")
user_product_features.head()

user_product_features: 8,474,661 pares x 7 colunas


,user_id,product_id,user_product_purchase_count,user_product_reorder_count,user_product_avg_add_to_cart_order,user_product_was_bought_before,user_product_orders_since_last_purchase
0,1,196,10,9,1.400000,1,0
1,1,10258,9,8,3.333333,1,0
2,1,10326,1,0,5.000000,1,5
3,1,12427,10,9,3.300000,1,0
4,1,13032,3,2,6.333333,1,0


In [11]:
prior_orders_timeline = (
    df_prior_ordered[
        [
            "user_id",
            "order_id",
            "order_number",
            "days_since_prior_order",
        ]
    ]
    .drop_duplicates()
    .sort_values(["user_id", "order_number"])
    .copy()
)

prior_orders_timeline["days_since_prior_order_for_cumsum"] = (
    prior_orders_timeline["days_since_prior_order"].fillna(0)
)

prior_orders_timeline["user_prior_cumulative_days"] = (
    prior_orders_timeline
    .groupby("user_id")["days_since_prior_order_for_cumsum"]
    .cumsum()
)

user_last_prior_day = (
    prior_orders_timeline
    .groupby("user_id", as_index=False)
    .agg(user_last_prior_cumulative_day=("user_prior_cumulative_days", "max"))
)

product_last_purchase_day = (
    df_prior_ordered[["user_id", "product_id", "order_id", "order_number"]]
    .merge(
        prior_orders_timeline[
            ["user_id", "order_id", "user_prior_cumulative_days"]
        ],
        on=["user_id", "order_id"],
        how="left",
    )
    .sort_values(["user_id", "product_id", "order_number"])
    .groupby(["user_id", "product_id"], as_index=False)
    .tail(1)
    [
        [
            "user_id",
            "product_id",
            "user_prior_cumulative_days",
        ]
    ]
    .rename(
        columns={
            "user_prior_cumulative_days": "user_product_last_purchase_cumulative_day"
        }
    )
)

user_product_recency_days = (
    product_last_purchase_day
    .merge(
        user_last_prior_day,
        on="user_id",
        how="left",
    )
)

user_product_recency_days["user_product_days_since_last_purchase"] = (
    user_product_recency_days["user_last_prior_cumulative_day"]
    - user_product_recency_days["user_product_last_purchase_cumulative_day"]
)

user_product_recency_days = user_product_recency_days[
    [
        "user_id",
        "product_id",
        "user_product_days_since_last_purchase",
    ]
]

assert not user_product_recency_days.duplicated(subset=["user_id", "product_id"]).any(), (
    "user_product_recency_days contém pares duplicados."
)
assert user_product_recency_days["user_product_days_since_last_purchase"].ge(0).all(), (
    "days_since_last_purchase contém valores negativos."
)

print(f"user_product_recency_days: {user_product_recency_days.shape[0]:,} pares")
user_product_recency_days.head()

user_product_recency_days: 8,474,661 pares


,user_id,product_id,user_product_days_since_last_purchase
0,1,196,0.0
1,1,10258,0.0
2,1,10326,83.0
3,1,12427,0.0
4,1,13032,0.0


In [12]:
user_product_features = user_product_features.merge(
    user_product_recency_days,
    on=["user_id", "product_id"],
    how="inner",
)

assert not user_product_features.duplicated(subset=["user_id", "product_id"]).any(), (
    "user_product_features contém pares duplicados após merge de recência."
)
assert user_product_features["user_product_days_since_last_purchase"].notna().all(), (
    "Existem valores nulos em user_product_days_since_last_purchase."
)

print(f"user_product_features: {user_product_features.shape[0]:,} pares x {user_product_features.shape[1]:,} colunas")
user_product_features.head()

user_product_features: 8,474,661 pares x 8 colunas


,user_id,product_id,user_product_purchase_count,user_product_reorder_count,user_product_avg_add_to_cart_order,user_product_was_bought_before,user_product_orders_since_last_purchase,user_product_days_since_last_purchase
0,1,196,10,9,1.400000,1,0,0.0
1,1,10258,9,8,3.333333,1,0,0.0
2,1,10326,1,0,5.000000,1,5,83.0
3,1,12427,10,9,3.300000,1,0,0.0
4,1,13032,3,2,6.333333,1,0,0.0


---

## 6. Features de categoria

Esta seção calcula features históricas de preferência do usuário por categoria, usando exclusivamente o conjunto `prior`.

As features conectam cada usuário aos aisles e departments dos produtos candidatos. Isso permite que o modelo diferencie produtos novos para o usuário, mas pertencentes a categorias já frequentes em seu histórico.

Também são preservados os identificadores categóricos do produto, que podem ser usados como features tabulares ou como base para análises futuras.

In [13]:
product_category_features = (
    df_prior
    .groupby("product_id", as_index=False)
    .agg(
        aisle_id=("aisle_id", "first"),
        department_id=("department_id", "first"),
    )
)

assert product_category_features["product_id"].is_unique, (
    "product_category_features contém produtos duplicados."
)
assert product_category_features["aisle_id"].notna().all(), (
    "Existem produtos sem aisle_id."
)
assert product_category_features["department_id"].notna().all(), (
    "Existem produtos sem department_id."
)

print(f"product_category_features: {product_category_features.shape[0]:,} produtos x {product_category_features.shape[1]:,} colunas")
product_category_features.head()

product_category_features: 49,677 produtos x 3 colunas


,product_id,aisle_id,department_id
0,1,61,19
1,2,104,13
2,3,94,7
3,4,38,1
4,5,5,13


In [14]:
user_aisle_features = (
    df_prior[df_prior["user_id"].isin(eligible_users)]
    .groupby(["user_id", "aisle_id"], as_index=False)
    .agg(
        user_aisle_purchase_count=("order_id", "size"),
    )
)

assert not user_aisle_features.duplicated(
    subset=["user_id", "aisle_id"]
).any(), "user_aisle_features contém pares user_id-aisle_id duplicados."

assert user_aisle_features["user_aisle_purchase_count"].gt(0).all(), (
    "Existem contagens inválidas em user_aisle_purchase_count."
)

print(f"user_aisle_features: {user_aisle_features.shape[0]:,} pares usuário-aisle")
user_aisle_features.head()

user_aisle_features: 3,647,699 pares usuário-aisle


,user_id,aisle_id,user_aisle_purchase_count
0,1,21,8
1,1,23,12
2,1,24,5
3,1,45,1
4,1,53,2


In [15]:
user_department_features = (
    df_prior[df_prior["user_id"].isin(eligible_users)]
    .groupby(["user_id", "department_id"], as_index=False)
    .agg(
        user_department_purchase_count=("order_id", "size"),
    )
)

assert not user_department_features.duplicated(
    subset=["user_id", "department_id"]
).any(), "user_department_features contém pares user_id-department_id duplicados."

assert user_department_features["user_department_purchase_count"].gt(0).all(), (
    "Existem contagens inválidas em user_department_purchase_count."
)

print(f"user_department_features: {user_department_features.shape[0]:,} pares usuário-department")
user_department_features.head()

user_department_features: 1,421,165 pares usuário-department


,user_id,department_id,user_department_purchase_count
0,1,4,5
1,1,7,13
2,1,13,1
3,1,14,3
4,1,16,13


---

## 7. Consolidação do dataset

Esta seção consolida o dataset final de modelagem em nível `user_id-product_id`.

O processo parte dos candidatos com target já construído e adiciona, de forma incremental, as features de usuário, produto, usuário-produto e categoria.

Valores ausentes são tratados explicitamente após a inspeção, separando ausência estrutural de informação histórica de possíveis problemas de merge.

### 7.1 Merge incremental das features

In [ ]:
CHUNK_USER_COUNT = 2_500
OVERWRITE_STAGED_DATASET = False

if STAGED_MODELING_DATASET_DIR.exists():
    existing_parts = list(STAGED_MODELING_DATASET_DIR.glob("*.parquet")) # Busca todos arquivos parquet no dir
    
    if existing_parts:
        if OVERWRITE_STAGED_DATASET:
            print(f"Limpando {len(existing_parts)} arquivos antigos em {STAGED_MODELING_DATASET_DIR}\n")
            for part_file in existing_parts:
                part_file.unlink() # O .unlink() deleta o arquivo fisicamente
        
        else:
            raise AssertionError(
                f"Diretório intermediário já contém arquivos e OVERWRITE_STAGED_DATASET é False: {STAGED_MODELING_DATASET_DIR} "
            )
else:
    STAGED_MODELING_DATASET_DIR.mkdir(parents=True, exist_ok=True)
    

eligible_user_array = np.array(sorted(eligible_users))
n_chunks = ceil(len(eligible_user_array) / CHUNK_USER_COUNT)
user_chunks = np.array_split(eligible_user_array, n_chunks)

print(f"Usuários elegíveis: {len(eligible_user_array):,}")
print(f"Chunks planejados: {len(user_chunks):,}")
print(f"Usuários por chunk: aproximadamente {CHUNK_USER_COUNT:,}")
print(f"Diretório intermediário: {STAGED_MODELING_DATASET_DIR}")

Limpando 53 arquivos antigos em /Users/helio/Library/CloudStorage/OneDrive-Pessoal/Pos_FIAP/Tech_Challanges/Fase2-Big-Data-Architecture/mlp-market-recommender-system/data/features/modeling_dataset_v2_raw

Usuários elegíveis: 131,209
Chunks planejados: 53
Usuários por chunk: aproximadamente 2,500
Diretório intermediário: /Users/helio/Library/CloudStorage/OneDrive-Pessoal/Pos_FIAP/Tech_Challanges/Fase2-Big-Data-Architecture/mlp-market-recommender-system/data/features/modeling_dataset_v2_raw


In [17]:
partition_stats = []

for chunk_idx, user_chunk in enumerate(user_chunks):
    chunk_candidates = modeling_df[
        modeling_df["user_id"].isin(user_chunk)
    ].copy()

    initial_rows = len(chunk_candidates)

    chunk_df = chunk_candidates.merge(
        user_features,
        on="user_id",
        how="left",
    )

    chunk_df = chunk_df.merge(
        product_features,
        on="product_id",
        how="left",
    )

    chunk_df = chunk_df.merge(
        user_product_features,
        on=["user_id", "product_id"],
        how="left",
    )

    chunk_df = chunk_df.merge(
        product_category_features,
        on="product_id",
        how="left",
    )

    chunk_df = chunk_df.merge(
        user_aisle_features,
        on=["user_id", "aisle_id"],
        how="left",
    )

    chunk_df = chunk_df.merge(
        user_department_features,
        on=["user_id", "department_id"],
        how="left",
    )

    assert len(chunk_df) == initial_rows, (
        f"Chunk {chunk_idx} teve alteração no número de linhas após merges."
    )
    assert not chunk_df.duplicated(subset=["user_id", "product_id"]).any(), (
        f"Chunk {chunk_idx} contém pares user_id-product_id duplicados."
    )

    part_path = STAGED_MODELING_DATASET_DIR / f"part-{chunk_idx:04d}.parquet"
    chunk_df.to_parquet(part_path, index=False)

    partition_stats.append(
        {
            "chunk_idx": chunk_idx,
            "rows": len(chunk_df),
            "users": chunk_df["user_id"].nunique(),
            "positives": int(chunk_df["target"].sum()),
            "path": str(part_path),
        }
    )
    
    if (chunk_idx + 1) % 5 == 0 or chunk_idx == 0:
        print(f"finalizado merge do dataset {chunk_idx + 1}/{n_chunks}")

    del chunk_candidates, chunk_df
    gc.collect()

partition_stats_df = pd.DataFrame(partition_stats)

print(f"\nPartições salvas: {len(partition_stats_df):,}")
print(f"Linhas salvas: {partition_stats_df['rows'].sum():,}")
print(f"Positivos salvos: {partition_stats_df['positives'].sum():,}")

finalizado merge do dataset 1/53
finalizado merge do dataset 5/53
finalizado merge do dataset 10/53
finalizado merge do dataset 15/53
finalizado merge do dataset 20/53
finalizado merge do dataset 25/53
finalizado merge do dataset 30/53
finalizado merge do dataset 35/53
finalizado merge do dataset 40/53
finalizado merge do dataset 45/53
finalizado merge do dataset 50/53

Partições salvas: 53
Linhas salvas: 45,519,000
Positivos salvos: 991,122


In [18]:
assert partition_stats_df["rows"].sum() == len(modeling_df), (
    "A soma das linhas particionadas difere do modeling_df original."
)
assert partition_stats_df["positives"].sum() == modeling_df["target"].sum(), (
    "A soma dos positivos particionados difere do modeling_df original."
)

partition_stats_df.head()

,chunk_idx,rows,users,positives,path
0,0,859182,2476,18605,/Users/helio/Library/CloudStorage/OneDrive-Pes...
1,1,859839,2476,18544,/Users/helio/Library/CloudStorage/OneDrive-Pes...
2,2,858714,2476,18608,/Users/helio/Library/CloudStorage/OneDrive-Pes...
3,3,859277,2476,18898,/Users/helio/Library/CloudStorage/OneDrive-Pes...
4,4,858765,2476,18758,/Users/helio/Library/CloudStorage/OneDrive-Pes...


---

## 8. Missing values e tratamentos

Esta seção trata valores ausentes gerados por ausência estrutural de histórico.

Os nulos esperados vêm principalmente de produtos candidatos que o usuário nunca comprou antes ou de categorias que ainda não aparecem no histórico do usuário.

O tratamento é feito por partição para evitar carregar o dataset completo em memória.

In [ ]:
OVERWRITE_MODELING_DATASET = False

if MODELING_DATASET_DIR.exists():
    existing_final_parts = list(MODELING_DATASET_DIR.glob("*.parquet"))

    if existing_final_parts:
        if OVERWRITE_MODELING_DATASET:
            print(
                f"Limpando {len(existing_final_parts)} arquivos antigos em "
                f"{MODELING_DATASET_DIR}"
            )
            for part_file in existing_final_parts:
                part_file.unlink()
        else:
            raise AssertionError(
                "Diretório final já contém arquivos e "
                f"OVERWRITE_MODELING_DATASET é False: {MODELING_DATASET_DIR}"
            )
else:
    MODELING_DATASET_DIR.mkdir(parents=True, exist_ok=True)

staged_part_paths = sorted(STAGED_MODELING_DATASET_DIR.glob("*.parquet"))

assert staged_part_paths, (
    f"Nenhuma partição encontrada em: {STAGED_MODELING_DATASET_DIR}"
)

print(f"Partições staged encontradas: {len(staged_part_paths):,}")
print(f"Dataset final tratado será salvo em: {MODELING_DATASET_DIR}")

Limpando 53 arquivos antigos em /Users/helio/Library/CloudStorage/OneDrive-Pessoal/Pos_FIAP/Tech_Challanges/Fase2-Big-Data-Architecture/mlp-market-recommender-system/data/features/modeling_dataset_v2
Partições staged encontradas: 53
Dataset final tratado será salvo em: /Users/helio/Library/CloudStorage/OneDrive-Pessoal/Pos_FIAP/Tech_Challanges/Fase2-Big-Data-Architecture/mlp-market-recommender-system/data/features/modeling_dataset_v2


In [20]:
null_stats = []

for part_idx, part_path in enumerate(staged_part_paths):
    part_df = pd.read_parquet(part_path)

    part_null_counts = part_df.isna().sum()

    null_stats.append(
        {
            "part": part_path.name,
            "rows": len(part_df),
            "columns_with_nulls": int(part_null_counts.gt(0).sum()),
            "total_nulls": int(part_null_counts.sum()),
        }
    )

    del part_df
    gc.collect()

null_stats_df = pd.DataFrame(null_stats)

print(f"Partições avaliadas: {len(null_stats_df):,}")
print(f"Total de nulos encontrados: {null_stats_df['total_nulls'].sum():,}")
print(f"Partições com nulos: {null_stats_df['total_nulls'].gt(0).sum():,}")


Partições avaliadas: 53
Total de nulos encontrados: 244,696,429
Partições com nulos: 53


In [21]:
null_counts_by_column = None

for part_path in staged_part_paths:
    part_df = pd.read_parquet(part_path)
    part_null_counts = part_df.isna().sum()

    if null_counts_by_column is None:
        null_counts_by_column = part_null_counts
    else:
        null_counts_by_column = null_counts_by_column.add(
            part_null_counts,
            fill_value=0,
        )

    del part_df
    gc.collect()

null_counts_by_column = (
    null_counts_by_column
    .astype("int64")
    .sort_values(ascending=False)
)

null_counts_by_column[null_counts_by_column > 0]

user_product_days_since_last_purchase      37450475
user_product_orders_since_last_purchase    37450475
user_product_was_bought_before             37450475
user_product_avg_add_to_cart_order         37450475
user_product_reorder_count                 37450475
user_product_purchase_count                37450475
user_aisle_purchase_count                  14905906
user_department_purchase_count              5087673
dtype: int64

In [22]:
treated_partition_stats = []

for part_idx, part_path in enumerate(staged_part_paths):
    part_df = pd.read_parquet(part_path)

    initial_rows = len(part_df)

    part_df["user_product_days_since_last_purchase"] = (
        part_df["user_product_days_since_last_purchase"]
        .fillna(
            part_df["user_avg_days_between_orders"]
            * part_df["user_prior_order_count"]
        )
    )
    
    part_df["user_product_orders_since_last_purchase"] = (
        part_df["user_product_orders_since_last_purchase"]
        .fillna(part_df["user_prior_order_count"])
    )
    
    part_df["user_product_was_bought_before"] = (
        part_df["user_product_was_bought_before"].fillna(0)
    )
    
    part_df["user_product_avg_add_to_cart_order"] = (
        part_df["user_product_avg_add_to_cart_order"].fillna(0)
    )
    
    part_df["user_product_reorder_count"] = (
        part_df["user_product_reorder_count"].fillna(0)
    )

    part_df["user_product_purchase_count"] = (
        part_df["user_product_purchase_count"].fillna(0)
    )

    part_df["user_aisle_purchase_count"] = (
        part_df["user_aisle_purchase_count"].fillna(0)
    )
    
    part_df["user_department_purchase_count"] = (
        part_df["user_department_purchase_count"].fillna(0)
    )
    

    assert len(part_df) == initial_rows, (
        f"Partição {part_path.name} alterou o número de linhas após tratamento."
    )
    assert not part_df.duplicated(subset=["user_id", "product_id"]).any(), (
        f"Partição {part_path.name} contém pares duplicados após tratamento."
    )
    assert not part_df.isna().any().any(), (
        f"Partição {part_path.name} ainda contém valores nulos."
    )

    final_part_path = MODELING_DATASET_DIR / part_path.name
    part_df.to_parquet(final_part_path, index=False)

    treated_partition_stats.append(
        {
            "part": part_path.name,
            "rows": len(part_df),
            "users": part_df["user_id"].nunique(),
            "positives": int(part_df["target"].sum()),
            "path": str(final_part_path),
        }
    )

    if (part_idx + 1) % 5 == 0 or part_idx == 0:
        print(f"Tratamento finalizado: {part_idx + 1}/{len(staged_part_paths)}")

    del part_df
    gc.collect()

treated_partition_stats_df = pd.DataFrame(treated_partition_stats)

print(f"\nPartições finais salvas: {len(treated_partition_stats_df):,}")
print(f"Linhas finais: {treated_partition_stats_df['rows'].sum():,}")
print(f"Positivos finais: {treated_partition_stats_df['positives'].sum():,}")

Tratamento finalizado: 1/53
Tratamento finalizado: 5/53
Tratamento finalizado: 10/53
Tratamento finalizado: 15/53
Tratamento finalizado: 20/53
Tratamento finalizado: 25/53
Tratamento finalizado: 30/53
Tratamento finalizado: 35/53
Tratamento finalizado: 40/53
Tratamento finalizado: 45/53
Tratamento finalizado: 50/53

Partições finais salvas: 53
Linhas finais: 45,519,000
Positivos finais: 991,122


In [23]:
assert treated_partition_stats_df["rows"].sum() == len(modeling_df), (
    "A soma das linhas finais difere do modeling_df original."
)
assert treated_partition_stats_df["positives"].sum() == modeling_df["target"].sum(), (
    "A soma dos positivos finais difere do modeling_df original."
)

final_null_total = 0

for part_path in sorted(MODELING_DATASET_DIR.glob("*.parquet")):
    part_df = pd.read_parquet(part_path)
    final_null_total += int(part_df.isna().sum().sum())

    del part_df
    gc.collect()

assert final_null_total == 0, (
    f"Dataset final ainda contém {final_null_total:,} valores nulos."
)

print("Dataset final sem valores nulos.")
print(f"Linhas validadas: {treated_partition_stats_df['rows'].sum():,}")
print(f"Positivos validados: {treated_partition_stats_df['positives'].sum():,}")

Dataset final sem valores nulos.
Linhas validadas: 45,519,000
Positivos validados: 991,122


---

## 9. Validação anti-leakage

Esta seção valida que o dataset final preserva a separação temporal definida para o projeto.

As features devem ser calculadas exclusivamente a partir do histórico `prior`.

O conjunto `train` só pode aparecer como target supervisionado já produzido no notebook `03-candidate-strategy.ipynb`.

In [24]:
final_part_paths = sorted(MODELING_DATASET_DIR.glob("*.parquet"))

assert final_part_paths, (
    f"Nenhuma partição final encontrada em: {MODELING_DATASET_DIR}"
)

train_order_ids = set(df.loc[df["eval_set"] == "train", "order_id"].unique())
prior_order_ids = set(df.loc[df["eval_set"] == "prior", "order_id"].unique())

assert train_order_ids, "Nenhum order_id de train encontrado no dataset unificado."
assert prior_order_ids, "Nenhum order_id de prior encontrado no dataset unificado."
assert train_order_ids.isdisjoint(prior_order_ids), (
    "Há interseção entre order_id de prior e train."
)

print(f"Partições finais: {len(final_part_paths):,}")
print(f"Pedidos prior: {len(prior_order_ids):,}")
print(f"Pedidos train: {len(train_order_ids):,}")

Partições finais: 53
Pedidos prior: 3,214,874
Pedidos train: 131,209


In [25]:
assert df_prior["eval_set"].eq("prior").all(), (
    "df_prior contém registros fora do conjunto prior."
)
assert df_prior_ordered["eval_set"].eq("prior").all(), (
    "df_prior_ordered contém registros fora do conjunto prior."
)
assert set(prior_orders_timeline["order_id"]).isdisjoint(train_order_ids), (
    "prior_orders_timeline contém order_id do conjunto train."
)
assert set(df_prior_ordered["order_id"]).isdisjoint(train_order_ids), (
    "df_prior_ordered contém order_id do conjunto train."
)

print("Fontes de features validadas: apenas registros prior.")

Fontes de features validadas: apenas registros prior.


In [26]:
forbidden_final_columns = {
    "order_id",
    "eval_set",
    "order_number",
    "order_dow",
    "order_hour_of_day",
    "days_since_prior_order",
    "product_name",
    "aisle",
    "department",
}

final_columns = set()

for part_path in final_part_paths:
    parquet_file = pq.ParquetFile(part_path)
    final_columns.update(parquet_file.schema.names) # Evita ler dados, pega so schema

leaked_columns = forbidden_final_columns.intersection(final_columns)

assert not leaked_columns, (
    f"Dataset final contém colunas transacionais não permitidas: {sorted(leaked_columns)}"
)

print(f"Colunas finais validadas: {len(final_columns):,}")
print("Nenhuma coluna transacional proibida foi encontrada.")

Colunas finais validadas: 22
Nenhuma coluna transacional proibida foi encontrada.


In [27]:
anti_leakage_stats = []

for part_idx, part_path in enumerate(final_part_paths):
    part_df = pd.read_parquet(
        part_path,
        columns=["user_id", "product_id", "target"],
    )

    assert not part_df.duplicated(subset=["user_id", "product_id"]).any(), (
        f"Partição {part_path.name} contém pares user_id-product_id duplicados."
    )

    part_users = part_df["user_id"].unique()

    expected_targets = modeling_df.loc[
        modeling_df["user_id"].isin(part_users),
        ["user_id", "product_id", "target"],
    ]

    target_check = part_df.merge(
        expected_targets,
        on=["user_id", "product_id"],
        how="left",
        suffixes=("_final", "_original"),
    )

    assert len(target_check) == len(part_df), (
        f"Validação de target alterou linhas na partição {part_path.name}."
    )
    assert target_check["target_original"].notna().all(), (
        f"Partição {part_path.name} contém par ausente nos candidatos originais."
    )
    assert (
        target_check["target_final"].to_numpy()
        == target_check["target_original"].to_numpy()
    ).all(), (
        f"Target final difere do target original na partição {part_path.name}."
    )

    anti_leakage_stats.append(
        {
            "part": part_path.name,
            "rows": len(part_df),
            "users": part_df["user_id"].nunique(),
            "positives": int(part_df["target"].sum()),
        }
    )

    if (part_idx + 1) % 5 == 0 or part_idx == 0:
        print(f"Validação anti-leakage finalizada: {part_idx + 1}/{len(final_part_paths)}")

    del part_df, expected_targets, target_check
    gc.collect()

anti_leakage_stats_df = pd.DataFrame(anti_leakage_stats)

print(f"\nPartições validadas: {len(anti_leakage_stats_df):,}")
print(f"Linhas validadas: {anti_leakage_stats_df['rows'].sum():,}")
print(f"Positivos validados: {anti_leakage_stats_df['positives'].sum():,}")

Validação anti-leakage finalizada: 1/53
Validação anti-leakage finalizada: 5/53
Validação anti-leakage finalizada: 10/53
Validação anti-leakage finalizada: 15/53
Validação anti-leakage finalizada: 20/53
Validação anti-leakage finalizada: 25/53
Validação anti-leakage finalizada: 30/53
Validação anti-leakage finalizada: 35/53
Validação anti-leakage finalizada: 40/53
Validação anti-leakage finalizada: 45/53
Validação anti-leakage finalizada: 50/53

Partições validadas: 53
Linhas validadas: 45,519,000
Positivos validados: 991,122


In [28]:
assert anti_leakage_stats_df["rows"].sum() == len(modeling_df), (
    "Total de linhas finais difere do total de candidatos original."
)
assert anti_leakage_stats_df["positives"].sum() == modeling_df["target"].sum(), (
    "Total de positivos finais difere do target original."
)

print("Validação anti-leakage concluída.")
print("Features calculadas exclusivamente a partir do prior.")
print("Target preservado a partir do notebook 03.")
print("Unicidade user_id-product_id preservada no dataset final.")

Validação anti-leakage concluída.
Features calculadas exclusivamente a partir do prior.
Target preservado a partir do notebook 03.
Unicidade user_id-product_id preservada no dataset final.


---

## 10. Persistência

Esta seção valida o artefato final salvo em Parquet particionado e cria o arquivo de metadados das features.

As validações são feitas sem carregar o dataset completo em memória.

In [29]:
final_part_paths = sorted(MODELING_DATASET_DIR.glob("*.parquet"))

assert MODELING_DATASET_DIR.exists(), (
    f"Diretório final não encontrado: {MODELING_DATASET_DIR}"
)
assert final_part_paths, (
    f"Nenhuma partição Parquet encontrada em: {MODELING_DATASET_DIR}"
)

print(f"Diretório final: {MODELING_DATASET_DIR}")
print(f"Partições finais: {len(final_part_paths):,}")

Diretório final: /Users/helio/Library/CloudStorage/OneDrive-Pessoal/Pos_FIAP/Tech_Challanges/Fase2-Big-Data-Architecture/mlp-market-recommender-system/data/features/modeling_dataset_v2
Partições finais: 53


In [30]:
modeling_dataset = ds.dataset(
    MODELING_DATASET_DIR,
    format="parquet",
)

dataset_schema = modeling_dataset.schema
dataset_columns = dataset_schema.names

print(f"Colunas no dataset final: {len(dataset_columns):,}")
print(dataset_columns)

Colunas no dataset final: 22
['user_id', 'product_id', 'target', 'user_prior_order_count', 'user_avg_basket_size', 'user_avg_days_between_orders', 'user_reorder_rate', 'user_has_single_prior_order', 'product_prior_purchase_count', 'product_prior_reorder_rate', 'product_popularity_pct', 'product_popularity_log', 'user_product_purchase_count', 'user_product_reorder_count', 'user_product_avg_add_to_cart_order', 'user_product_was_bought_before', 'user_product_orders_since_last_purchase', 'user_product_days_since_last_purchase', 'aisle_id', 'department_id', 'user_aisle_purchase_count', 'user_department_purchase_count']


In [31]:
persisted_row_count = modeling_dataset.count_rows()

persisted_positive_count = 0

target_scanner = modeling_dataset.scanner(
    columns=["target"],
    batch_size=1_000_000,
)

for batch in target_scanner.to_batches():
    persisted_positive_count += pc.sum(batch.column("target")).as_py()

assert persisted_row_count == len(modeling_df), (
    "Total de linhas persistidas difere do modeling_df original."
)
assert persisted_positive_count == modeling_df["target"].sum(), (
    "Total de positivos persistidos difere do target original."
)

print(f"Linhas persistidas: {persisted_row_count:,}")
print(f"Positivos persistidos: {persisted_positive_count:,}")

Linhas persistidas: 45,519,000
Positivos persistidos: 991,122


In [32]:
print("Persistência final validada.")
print(f"Dataset final: {MODELING_DATASET_DIR}")

Persistência final validada.
Dataset final: /Users/helio/Library/CloudStorage/OneDrive-Pessoal/Pos_FIAP/Tech_Challanges/Fase2-Big-Data-Architecture/mlp-market-recommender-system/data/features/modeling_dataset_v2
